<a href="https://colab.research.google.com/github/AmanuelDaget/Incremental_Map_Reduce/blob/main/Big_Data_Incremental_Map_Reduce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Incremental Map Reduce**

**Import Libraries**

In [3]:
import pandas as pd
import numpy as np
import time
import json
import copy
import os
from collections import defaultdict

**CORE MapReduce FRAMEWORK**

In [4]:
class MapReduceResult:
    """Container for a MapReduce job result with timing metadata."""
    def __init__(self, output, elapsed_ms, records_processed, method):
        self.output = output
        self.elapsed_ms = elapsed_ms
        self.records_processed = records_processed
        self.method = method

    def __repr__(self):
        return (f"MapReduceResult(method={self.method}, "
                f"records={self.records_processed}, "
                f"time={self.elapsed_ms:.2f}ms)")


**  STANDARD BATCH MapReduce (Baseline)**

In [5]:
class BatchMapReduce:
    """
    Traditional batch MapReduce that reprocesses ALL data on every update.
    Used as baseline for performance comparison.
    """

    def run(self, full_data: pd.DataFrame, mapper_fn, reducer_fn,
            task_name="Task"):
        t0 = time.perf_counter()
        # MAP phase
        mapped = []
        for _, row in full_data.iterrows():
            pairs = mapper_fn(row)
            mapped.extend(pairs)

        # SHUFFLE & SORT
        grouped = defaultdict(list)
        for k, v in mapped:
            grouped[k].append(v)

        # REDUCE phase
        output = {}
        for key, values in grouped.items():
            output[key] = reducer_fn(key, values)

        elapsed_ms = (time.perf_counter() - t0) * 1000
        return MapReduceResult(output, elapsed_ms,
                               len(full_data), "BatchMapReduce")

**i2MapReduce: INCREMENTAL MapReduce WITH CPC**

In [6]:
class i2MapReduce:
    """
    Incremental MapReduce (i2MapReduce) with Change Propagation Control (CPC).

    Core idea: Reuse converged intermediate state from the previous batch.
    Only propagate changes caused by new/modified records through the
    Map → Shuffle → Reduce pipeline.

    CPC Mechanism:
      - Maintains a "converged state" (CS) for each key from previous batch
      - For each new record, computes the DELTA from the previous contribution
      - Only keys with non-trivial deltas are re-reduced
      - Merges delta output with converged state to produce final result
    """

    def __init__(self):
        # Converged State: key → previous reduce output
        self.converged_state = {}
        # Previous intermediate map outputs: record_id → list of (key, value)
        self.prev_mapped = {}
        # Previous grouped values: key → list of values
        self.prev_grouped = defaultdict(list)

    def _compute_delta(self, new_mapped_pairs, record_id):
        """
        CPC: Compute the difference between new and old mapped output
        for a given record. Returns (added_pairs, removed_pairs).
        """
        new_set = {}
        for k, v in new_mapped_pairs:
            if k not in new_set:
                new_set[k] = []
            new_set[k].append(v)

        old_pairs = self.prev_mapped.get(record_id, [])
        old_set = {}
        for k, v in old_pairs:
            if k not in old_set:
                old_set[k] = []
            old_set[k].append(v)

        added = []
        removed = []
        all_keys = set(new_set) | set(old_set)
        for k in all_keys:
            n_vals = new_set.get(k, [])
            o_vals = old_set.get(k, [])
            # Values added: in new but not in old
            for v in n_vals:
                if v not in o_vals:
                    added.append((k, v))
            # Values removed: in old but not in new
            for v in o_vals:
                if v not in n_vals:
                    removed.append((k, v))

        return added, removed

    def run(self, new_batch: pd.DataFrame, mapper_fn, reducer_fn,
            incremental_reducer_fn=None, full_data_for_ids: pd.DataFrame = None,
            task_name="Task"):
        """
        Run incremental MapReduce on a new batch.

        Parameters
        ----------
        new_batch : DataFrame of newly arrived records
        mapper_fn : function(row) -> list of (key, value)
        reducer_fn : function(key, values) -> result (for full recompute)
        incremental_reducer_fn : function(key, old_result, added_values,
                                          removed_values) -> new_result
            If None, falls back to full re-reduce of affected keys only.
        full_data_for_ids : DataFrame containing all records (for ID lookup)
        """
        t0 = time.perf_counter()
        records_processed = 0
        affected_keys = set()

        # ── MAP phase: only new records ───────────────────────────────────
        for _, row in new_batch.iterrows():
            record_id = (row["country"], row["batch"])
            new_pairs = mapper_fn(row)
            records_processed += 1

            # CPC: compute delta from previous mapping
            added, removed = self._compute_delta(new_pairs, record_id)

            if added or removed:
                # Update grouped state
                for k, v in added:
                    self.prev_grouped[k].append(v)
                    affected_keys.add(k)
                for k, v in removed:
                    if v in self.prev_grouped[k]:
                        self.prev_grouped[k].remove(v)
                    affected_keys.add(k)

            # Update tracked mapping for this record
            self.prev_mapped[record_id] = new_pairs

        # ── REDUCE phase: only affected keys (CPC) ────────────────────────
        delta_output = {}
        for key in affected_keys:
            values = self.prev_grouped[key]
            if incremental_reducer_fn and key in self.converged_state:
                # Use incremental reducer (faster - avoids full recompute)
                delta_output[key] = incremental_reducer_fn(
                    key, self.converged_state[key], values)
            else:
                delta_output[key] = reducer_fn(key, values)

        # ── MERGE: converged state + delta ───────────────────────────────
        output = copy.copy(self.converged_state)
        output.update(delta_output)

        # Update converged state for next batch
        self.converged_state = copy.copy(output)

        elapsed_ms = (time.perf_counter() - t0) * 1000
        return MapReduceResult(output, elapsed_ms,
                               records_processed, "i2MapReduce")

**ANALYTICS TASKS**

In [7]:
# ── Task A: GDP Sum & Count per Region ───────────────────────────────────────

def gdp_mapper(row):
    """Emit (region, gdp_value) pair."""
    return [(row["region"], row["gdp_billion_usd"])]

def gdp_reducer(key, values):
    """Aggregate GDP values per region."""
    return {
        "total_gdp": round(sum(values), 2),
        "count": len(values),
        "avg_gdp": round(sum(values) / len(values), 2),
        "max_gdp": round(max(values), 2),
    }

def gdp_incremental_reducer(key, old_result, new_values):
    """Incremental version: update aggregation without full recompute."""
    return gdp_reducer(key, new_values)  # recompute from updated grouped values


# ── Task B: Average HDI per Region ───────────────────────────────────────────

def hdi_mapper(row):
    return [(row["region"], row["hdi"])]

def hdi_reducer(key, values):
    return {
        "avg_hdi": round(sum(values) / len(values), 4),
        "min_hdi": round(min(values), 4),
        "max_hdi": round(max(values), 4),
        "count":   len(values),
    }

def hdi_incremental_reducer(key, old_result, new_values):
    return hdi_reducer(key, new_values)


# ── Task C: GDP per Capita – Top countries ────────────────────────────────────

def gdp_per_cap_mapper(row):
    return [("global_ranking", (row["country"], row["gdp_per_capita_usd"]))]

def gdp_per_cap_reducer(key, values):
    """Sort countries by GDP per capita, return top-10."""
    sorted_vals = sorted(values, key=lambda x: x[1], reverse=True)
    return {"top_10": sorted_vals[:10]}

def gdp_per_cap_incremental_reducer(key, old_result, new_values):
    return gdp_per_cap_reducer(key, new_values)

**EXPERIMENT RUNNER**

In [12]:
def run_experiments(data_dir="/content/data"):
    """
    Compare BatchMapReduce vs i2MapReduce across 4 batches for 3 tasks.
    Returns performance metrics and results.
    """
    print("=" * 65)
    print("  i2MapReduce vs Batch MapReduce — Performance Experiment")
    print("=" * 65)

    # Load all batches
    batches = []
    for b in range(4):
        df = pd.read_csv(f"{data_dir}/batch_{b}.csv")
        batches.append(df)
        print(f"  Loaded Batch {b}: {len(df)} records (Year {2020+b})")

    tasks = [
        ("GDP Regional Aggregation", gdp_mapper, gdp_reducer, gdp_incremental_reducer),
        ("HDI Regional Averages",    hdi_mapper, hdi_reducer, hdi_incremental_reducer),
        ("GDP-per-capita Top-K",     gdp_per_cap_mapper, gdp_per_cap_reducer,
                                     gdp_per_cap_incremental_reducer),
    ]

    metrics = []
    all_results = {}

    for task_name, mapper, reducer, inc_reducer in tasks:
        print(f"\n{'─'*65}")
        print(f"  Task: {task_name}")
        print(f"{'─'*65}")

        batch_mr   = BatchMapReduce()
        i2_mr      = i2MapReduce()

        cumulative_data = pd.DataFrame()

        for batch_num, batch_df in enumerate(batches):
            # Accumulate data (batch MapReduce needs all data each time)
            cumulative_data = pd.concat(
                [cumulative_data, batch_df], ignore_index=True)

            # Batch MapReduce on ALL data
            batch_result = batch_mr.run(
                cumulative_data, mapper, reducer, task_name)

            # i2MapReduce on NEW data only
            i2_result = i2_mr.run(
                batch_df, mapper, reducer, inc_reducer,
                full_data_for_ids=cumulative_data, task_name=task_name)

            speedup = batch_result.elapsed_ms / max(i2_result.elapsed_ms, 0.001)
            data_reduction = (1 - i2_result.records_processed /
                              batch_result.records_processed) * 100

            print(f"  Batch {batch_num} | Batch: {batch_result.elapsed_ms:6.2f}ms "
                  f"({batch_result.records_processed} recs) | "
                  f"i2MR: {i2_result.elapsed_ms:6.2f}ms "
                  f"({i2_result.records_processed} recs) | "
                  f"Speedup: {speedup:.2f}x | "
                  f"Data reduction: {data_reduction:.1f}%")

            metrics.append({
                "task":              task_name,
                "batch":             batch_num,
                "batch_mr_ms":       round(batch_result.elapsed_ms, 4),
                "i2mr_ms":           round(i2_result.elapsed_ms, 4),
                "batch_records":     batch_result.records_processed,
                "i2mr_records":      i2_result.records_processed,
                "speedup":           round(speedup, 3),
                "data_reduction_pct": round(data_reduction, 2),
            })

            if batch_num == 3:
                all_results[task_name] = i2_result.output

    metrics_df = pd.DataFrame(metrics)
    print(f"\n{'='*65}")
    print("  SUMMARY: Average speedup per task")
    print(f"{'='*65}")
    summary = (metrics_df.groupby("task")[["speedup","data_reduction_pct"]]
               .mean().round(2))
    print(summary.to_string())

    return metrics_df, all_results


def print_final_results(all_results):
    print("\n" + "=" * 65)
    print("  FINAL ANALYTICS RESULTS (after all 4 batches)")
    print("=" * 65)

    if "GDP Regional Aggregation" in all_results:
        print("\n[A] GDP per Region (Billion USD, cumulative 2020-2023):")
        for region, stats in sorted(all_results["GDP Regional Aggregation"].items()):
            print(f"  {region:12s}: total={stats['total_gdp']:>10,.1f}B  "
                  f"avg={stats['avg_gdp']:>8,.1f}B  n={stats['count']}")

    if "HDI Regional Averages" in all_results:
        print("\n[B] Average HDI per Region:")
        for region, stats in sorted(all_results["HDI Regional Averages"].items()):
            print(f"  {region:12s}: avg={stats['avg_hdi']:.4f}  "
                  f"range=[{stats['min_hdi']:.3f}, {stats['max_hdi']:.3f}]  "
                  f"n={stats['count']}")

    if "GDP-per-capita Top-K" in all_results:
        print("\n[C] Top-10 Countries by GDP per Capita (USD, latest):")
        top10 = all_results["GDP-per-capita Top-K"].get("global_ranking", {})
        if "top_10" in top10:
            for rank, (country, gdppc) in enumerate(top10["top_10"], 1):
                print(f"  {rank:2d}. {country:20s}  ${gdppc:>12,.0f}")


if __name__ == "__main__":
    metrics_df, all_results = run_experiments()
    print_final_results(all_results)

    # Save metrics
    metrics_df.to_csv("/content/data/performance_metrics.csv", index=False)
    print("\nSaved metrics: /content/data/performance_metrics.csv")

  i2MapReduce vs Batch MapReduce — Performance Experiment
  Loaded Batch 0: 61 records (Year 2020)
  Loaded Batch 1: 61 records (Year 2021)
  Loaded Batch 2: 61 records (Year 2022)
  Loaded Batch 3: 61 records (Year 2023)

─────────────────────────────────────────────────────────────────
  Task: GDP Regional Aggregation
─────────────────────────────────────────────────────────────────
  Batch 0 | Batch:   3.01ms (61 recs) | i2MR:   3.39ms (61 recs) | Speedup: 0.89x | Data reduction: 0.0%
  Batch 1 | Batch:   5.36ms (122 recs) | i2MR:   3.26ms (61 recs) | Speedup: 1.64x | Data reduction: 50.0%
  Batch 2 | Batch:   7.73ms (183 recs) | i2MR:   4.06ms (61 recs) | Speedup: 1.90x | Data reduction: 66.7%
  Batch 3 | Batch:  10.48ms (244 recs) | i2MR:   3.25ms (61 recs) | Speedup: 3.23x | Data reduction: 75.0%

─────────────────────────────────────────────────────────────────
  Task: HDI Regional Averages
─────────────────────────────────────────────────────────────────
  Batch 0 | Batch:   2.